# Notebook: SSL + Semi-Supervised for Electricity-Theft Detection (SGCC)
Author: `<your name>`  
Colab-ready, deterministic seeds, robust loaders


In [ ]:

#@title 1) Environment & Imports
!pip -q install tensorflow==2.15.1 umap-learn reportlab

import os, glob, gc, math, warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_curve, precision_recall_fscore_support)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, regularizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


ERROR: Could not find a version that satisfies the requirement tensorflow==2.15.1 (from versions: 2.16.0rc0, 2.16.1, 2.16.2, 2.17.0rc0, 2.17.0rc1, 2.17.0, 2.17.1, 2.18.0rc0, 2.18.0rc1, 2.18.0rc2, 2.18.0, 2.18.1, 2.19.0rc0, 2.19.0, 2.19.1, 2.20.0rc0, 2.20.0)
ERROR: No matching distribution found for tensorflow==2.15.1
TensorFlow: 2.19.0


In [ ]:

#@title 2) Get Data (GitHub clone or manual upload)
REPO_URL = "https://github.com/henryRDlab/ElectricityTheftDetection.git"
REPO_DIR = "/content/ElectricityTheftDetection"

if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
    print("Repo cloned:", REPO_DIR)

# Search candidate CSV/TXT files
candidate_csvs = []
for pattern in [f"{REPO_DIR}/**/*.csv", f"{REPO_DIR}/**/*.txt", "/content/*.csv", "/content/*.txt"]:
    candidate_csvs.extend(glob.glob(pattern, recursive=True))

def sort_key(p):
    size = os.path.getsize(p)
    name = os.path.basename(p).lower()
    bonus = sum(kw in name for kw in ["cons","power","sgcc","theft","usage","load","data","flag"])
    return (-bonus, -size)

candidate_csvs = sorted(set(candidate_csvs), key=sort_key)
print("Found:", len(candidate_csvs), "candidates")
for i, p in enumerate(candidate_csvs[:12]):
    print(f"[{i}] {p} | {round(os.path.getsize(p)/1e6,2)} MB")


Repo cloned: /content/ElectricityTheftDetection
Found: 0 candidates


In [ ]:

#@title 3) Loader to standardize dataset to LONG format with labels (if exist)
import pandas as pd
import numpy as np

def infer_and_load(filepath, max_rows=None):
    # detect delimiter
    for sep in [",",";","\t","|"]:
        try:
            _ = pd.read_csv(filepath, nrows=1000, sep=sep)
            delimiter = sep; break
        except: pass
    else:
        delimiter = ","

    sample = pd.read_csv(filepath, nrows=5000, sep=delimiter)
    sample.columns = [c.lower().strip() for c in sample.columns]
    cols = sample.columns.tolist()

    def find(cands):
        for k in cands:
            for c in cols:
                if k in c:
                    return c
        return None

    cid  = find(["customer","cust","meter","user","account","consumer","id","cons_no"])
    cts  = find(["time","date","timestamp","day"])
    cval = find(["kwh","cons","usage","power","load","value","reading"])
    clab = find(["flag","label","target","class","theft"])

    # LONG case
    if cid and cts and cval:
        use = [cid, cts, cval] + ([clab] if clab else [])
        df  = pd.read_csv(filepath, usecols=use, sep=delimiter, nrows=max_rows)
        df.columns = ["customer_id","timestamp","consumption"] + (["label"] if clab else [])
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True).dt.tz_convert(None)
        df = df.dropna(subset=["timestamp"])
        return df

    # WIDE case: timestamps as columns or rows
    df_full = pd.read_csv(filepath, sep=delimiter, nrows=max_rows)
    df_full.columns = [c.lower().strip() for c in df_full.columns]

    if cid and cid in df_full.columns:
        pass
    else:
        df_full.insert(0, "customer_id", np.arange(len(df_full)))
        cid = "customer_id"

    # timestamps in columns?
    parsed = []
    for c in df_full.columns:
        if c == cid or c == clab: continue
        try:
            pd.to_datetime(c, errors="raise")
            parsed.append(c)
        except: pass

    if parsed:
        wide = df_full[[cid] + parsed + ([clab] if clab else [])].copy()
        long = wide.melt(id_vars=[cid] + ([clab] if clab else []), var_name="timestamp", value_name="consumption")
        long["timestamp"] = pd.to_datetime(long["timestamp"], errors="coerce")
        long.rename(columns={cid:"customer_id"}, inplace=True)
        long = long.dropna(subset=["timestamp"])
        return long

    # rows as timestamps (first column)
    ts_col = df_full.columns[1] if df_full.columns[0] == "customer_id" else df_full.columns[0]
    try:
        df_full[ts_col] = pd.to_datetime(df_full[ts_col], errors="coerce")
        long = df_full.melt(id_vars=[ts_col] + ([clab] if clab else []), var_name="customer_id", value_name="consumption")
        long.rename(columns={ts_col:"timestamp"}, inplace=True)
        long = long.dropna(subset=["timestamp"])
        return long
    except:
        raise RuntimeError("Could not infer dataset format. Please map columns manually.")

# Try to load first matching file
LONG = None
for f in candidate_csvs[:10]:
    try:
        LONG = infer_and_load(f)
        print("Loaded:", f, "| shape:", LONG.shape)
        break
    except Exception as e:
        print("Skip:", f, "|", e)

if LONG is None:
    print("No usable file detected. Upload your CSV now.")
    from google.colab import files
    uploaded = files.upload()
    f = list(uploaded.keys())[0]
    LONG = infer_and_load(f)

display(LONG.head())
print("Columns:", LONG.columns.tolist())


No usable file detected. Upload your CSV now.


Saving data.csv to data.csv


,customer_id,flag,timestamp,consumption
0,0387DD8A07E07FDA6271170F86AD9151,1,2014-01-01,NaN
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,1,2014-01-01,NaN
2,4B75AC4F2D8434CFF62DB64D0BB43103,1,2014-01-01,NaN
3,B32AC8CC6D5D805AC053557AB05F5343,1,2014-01-01,NaN
4,EDFC78B07BA2908B3395C4EB2304665E,1,2014-01-01,2.9


Columns: ['customer_id', 'flag', 'timestamp', 'consumption']


In [ ]:

#@title 4) Daily aggregation, imputation, z-score per customer
def to_daily_panel(df_long, max_customers=None):
    df = df_long.copy()
    keep = ["customer_id","timestamp","consumption"] + (["label"] if "label" in df.columns else [])
    df = df[keep].dropna(subset=["timestamp","consumption"])
    df["customer_id"] = df["customer_id"].astype(str)
    df["timestamp"]   = pd.to_datetime(df["timestamp"], errors="coerce", utc=True).dt.tz_convert(None)
    df["consumption"] = pd.to_numeric(df["consumption"], errors="coerce")
    df = df.dropna(subset=["timestamp","consumption"])

    agg = df.groupby(["customer_id", pd.Grouper(key="timestamp", freq="D")])["consumption"].sum().reset_index()
    if max_customers and agg["customer_id"].nunique() > max_customers:
        keep_ids = agg["customer_id"].drop_duplicates().sample(max_customers, random_state=SEED)
        agg = agg[agg["customer_id"].isin(keep_ids)]

    panel = agg.pivot(index="customer_id", columns="timestamp", values="consumption").sort_index(axis=1)
    return panel

PANEL = to_daily_panel(LONG, max_customers=None)
print("Panel shape:", PANEL.shape)
display(PANEL.iloc[:5, :7])

# impute per-customer
PANEL = PANEL.groupby(level=0).apply(lambda s: s.ffill(axis=1).bfill(axis=1)).droplevel(0, axis=0)
PANEL = PANEL.fillna(0.0)

X = PANEL.values
mu = X.mean(axis=1, keepdims=True); sd = X.std(axis=1, keepdims=True) + 1e-6
PANEL_Z = pd.DataFrame((X - mu)/sd, index=PANEL.index, columns=PANEL.columns)

print("Z-scored panel sample:")
display(PANEL_Z.iloc[:5, :7])

cust_labels = None
if "label" in LONG.columns:
    tmp = LONG[["customer_id","label"]].dropna()
    try:
        tmp["label"] = pd.to_numeric(tmp["label"], errors="coerce")
        tmp["label"] = (tmp["label"] > 0).astype(int)
    except:
        tmp["label"] = (
            tmp["label"].astype(str).str.lower().str.strip()
              .map({"1":"1","0":"0","true":"1","false":"0","theft":"1","normal":"0"})
              .fillna("0").astype(int)
        )
    cust_labels = tmp.groupby("customer_id")["label"].max().reset_index()
    print("Labels found. Pos rate:", cust_labels["label"].mean())
else:
    print("No labels column found.")


Panel shape: (42367, 1034)


timestamp,2014-01-01,2014-01-02,2014-01-03,2014-01-04,2014-01-05,2014-01-06,2014-01-07
customer_id,,,,,,,
00002188D4496609AC58502A1241C0E0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0000E78A22CB04533A0D9E1F2FBEEC5D,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0002D8E9C198E4A2B03BFA6D1E2E1B6D,8.04,5.94,11.96,5.09,4.72,7.21,8.47
000395F84A94D4CB2E5D4D77372CFB4D,22.23,22.46,20.13,21.79,24.00,24.00,19.10
0009290EB794D35944F1DD36FE324F3A,4.69,4.01,0.00,6.04,7.06,4.99,4.69


Z-scored panel sample:


timestamp,2014-01-01,2014-01-02,2014-01-03,2014-01-04,2014-01-05,2014-01-06,2014-01-07
customer_id,,,,,,,
00002188D4496609AC58502A1241C0E0,-0.567942,-0.567942,-0.567942,-0.567942,-0.567942,-0.567942,-0.567942
0000E78A22CB04533A0D9E1F2FBEEC5D,-0.378649,-0.378649,-0.378649,-0.378649,-0.378649,-0.378649,-0.378649
0002D8E9C198E4A2B03BFA6D1E2E1B6D,0.002294,-0.714354,1.340037,-1.004426,-1.130693,-0.280953,0.149036
000395F84A94D4CB2E5D4D77372CFB4D,-0.359585,-0.325622,-0.669681,-0.424557,-0.098217,-0.098217,-0.821776
0009290EB794D35944F1DD36FE324F3A,-0.382989,-0.576202,-1.715593,0.000596,0.290416,-0.297748,-0.382989


No labels column found.


In [ ]:
# Build cust_labels from 'flag' column
import pandas as pd
assert 'flag' in LONG.columns, "В LONG нет колонки 'flag'"

cust_labels = LONG[['customer_id', 'flag']].rename(columns={'flag':'label'}).dropna()
cust_labels['customer_id'] = cust_labels['customer_id'].astype(str)
cust_labels['label'] = pd.to_numeric(cust_labels['label'], errors='coerce').fillna(0).astype(int)
# на уровне клиента берём максимум (если встречается 1 хоть раз — считаем клиент меточным)
cust_labels = cust_labels.groupby('customer_id')['label'].max().reset_index()

# на всякий случай: привести индексы панели к str для корректной склейки
PANEL_Z.index = PANEL_Z.index.astype(str)

print("Labels ready. Pos rate:", cust_labels['label'].mean(), "| rows:", len(cust_labels))
cust_labels.head()


Labels ready. Pos rate: 0.08531577456811101 | rows: 42372


,customer_id,label
0,00002188D4496609AC58502A1241C0E0,0
1,0000E78A22CB04533A0D9E1F2FBEEC5D,0
2,0002D8E9C198E4A2B03BFA6D1E2E1B6D,0
3,000395F84A94D4CB2E5D4D77372CFB4D,1
4,0009290EB794D35944F1DD36FE324F3A,0


In [ ]:

#@title 5) Windowing utilities
WINDOW = 60  # days
STRIDE = 15

def build_windows(panel_z: pd.DataFrame, window=60, stride=15):
    X_windows, idx_map = [], []
    ts_cols = panel_z.columns.to_list()
    arr = panel_z.values
    for i, cid in enumerate(panel_z.index):
        row = arr[i]
        for s in range(0, row.shape[0]-window+1, stride):
            win = row[s:s+window]
            X_windows.append(win.astype(np.float32))
            idx_map.append((panel_z.index[i], ts_cols[s]))
    X = np.stack(X_windows) if X_windows else np.empty((0, window), dtype=np.float32)
    return X, idx_map


In [ ]:
#@title 6) Define denoising autoencoder (SSL)
from tensorflow import keras
from tensorflow.keras import layers, regularizers

def make_autoencoder(input_len, width=256, latent=64, dropout=0.2, l2=1e-6, noise_std=0.05):
    inp = layers.Input(shape=(input_len,))
    x = layers.GaussianNoise(stddev=noise_std)(inp)
    x = layers.Dense(width, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(width//2, activation="relu", kernel_regularizer=regularizers.l2(l2))(x)
    z = layers.Dense(latent, activation="relu", name="latent")(x)
    x = layers.Dense(width//2, activation="relu")(z)
    x = layers.Dense(width, activation="relu")(x)
    out = layers.Dense(input_len, activation="linear")(x)
    model = keras.Model(inp, out)
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
    return model


In [ ]:
# Посмотреть столбцы текущего LONG
print(LONG.columns.tolist())
LONG.head()


['customer_id', 'flag', 'timestamp', 'consumption']


,customer_id,flag,timestamp,consumption
0,0387DD8A07E07FDA6271170F86AD9151,1,2014-01-01,NaN
1,01D6177B5D4FFE0CABA9EF17DAFC2B84,1,2014-01-01,NaN
2,4B75AC4F2D8434CFF62DB64D0BB43103,1,2014-01-01,NaN
3,B32AC8CC6D5D805AC053557AB05F5343,1,2014-01-01,NaN
4,EDFC78B07BA2908B3395C4EB2304665E,1,2014-01-01,2.9


In [ ]:

#@title 7) Train AE on normal-only (if labels exist) and compute scores
if cust_labels is None:
    raise RuntimeError("Labels required for evaluation. If you truly have none, skip metrics cells or use synthetic labels.")

lab = cust_labels.copy()
lab["customer_id"] = lab["customer_id"].astype(str)
lab["label"] = lab["label"].astype(int)

normal_ids = lab[lab["label"]==0]["customer_id"]
PANEL_Z_normal = PANEL_Z.loc[PANEL_Z.index.isin(normal_ids)]

Xn, idx_n = build_windows(PANEL_Z_normal, window=WINDOW, stride=STRIDE)
print("Normal-only windows:", Xn.shape)

autoenc = make_autoencoder(WINDOW, width=256, latent=64, dropout=0.2, l2=1e-6, noise_std=0.05)
es  = callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)
rlr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1)

import numpy as np
perm = np.random.permutation(len(Xn)); split = int(0.9*len(Xn))
hist = autoenc.fit(Xn[perm[:split]], Xn[perm[:split]],
                   validation_data=(Xn[perm[split:]], Xn[perm[split:]]),
                   epochs=100, batch_size=256, callbacks=[es, rlr], verbose=2)

# Window scores for ALL customers
X_all, idx_all = build_windows(PANEL_Z, window=WINDOW, stride=STRIDE)
X_pred = autoenc.predict(X_all, verbose=0)
win_err = np.mean((X_pred - X_all)**2, axis=1)

df_err = pd.DataFrame(idx_all, columns=["customer_id","window_start"])
df_err["win_mse"] = win_err

# Client-level aggregators
agg = df_err.groupby("customer_id")["win_mse"].agg(
    p95=lambda s: np.quantile(s, 0.95),
    p99=lambda s: np.quantile(s, 0.99),
    max="max",
    mean="mean"
)

# Evaluate both NEGATED mean and NEGATED p99
A = agg.join(lab.set_index("customer_id"), how="inner").dropna()
y_true = A["label"].astype(int).values

from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, precision_recall_fscore_support

def eval_agg(col):
    s = -A[col].values  # NEGATED
    roc = roc_auc_score(y_true, s)
    pr  = average_precision_score(y_true, s)
    prec, rec, thr = precision_recall_curve(y_true, s)
    f1 = 2*(prec*rec)/(prec+rec+1e-12)
    best = np.nanargmax(f1)
    best_thr = thr[best-1] if 0<best<len(thr)+1 else np.quantile(s, 0.975)
    y_pred = (s >= best_thr).astype(int)
    P, R, F1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return dict(roc=roc, pr=pr, P=P, R=R, F1=F1, thr=float(best_thr))

m_res = eval_agg("mean"); p99_res = eval_agg("p99")
print("NEGATED mean  ->", m_res)
print("NEGATED p99   ->", p99_res)

# Save artifacts
os.makedirs("/content/figs", exist_ok=True)
pd.DataFrame(hist.history).to_csv("/content/ae_train_history.csv", index=False)
df_err.to_csv("/content/window_errors.csv", index=False)
agg.to_csv("/content/client_aggregates.csv")
with open("/content/ssl_eval.json","w") as f: json.dump({"mean":m_res,"p99":p99_res}, f, indent=2)

# Curves
plt.figure(figsize=(6,4))
plt.plot(hist.history["loss"], label="train"); plt.plot(hist.history["val_loss"], label="val")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title("AE Training"); plt.grid(True); plt.legend(); plt.tight_layout()
plt.savefig("/content/figs/training_curve.png", dpi=150); plt.close()

plt.figure(figsize=(6,4))
plt.hist(df_err["win_mse"], bins=60); plt.xlabel("Window MSE"); plt.ylabel("Count")
plt.title("Distribution of window errors"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/window_err_hist.png", dpi=150); plt.close()

plt.figure(figsize=(6,4))
plt.hist((-A["p99"]).values, bins=60); plt.xlabel("NEGATED p99 score"); plt.ylabel("Clients")
plt.title("Distribution of client scores"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/customer_score_hist.png", dpi=150); plt.close()


Normal-only windows: (2519075, 60)
Epoch 1/100
8857/8857 - 34s - 4ms/step - loss: 0.1043 - val_loss: 0.0628 - learning_rate: 1.0000e-03
Epoch 2/100
8857/8857 - 34s - 4ms/step - loss: 0.0785 - val_loss: 0.0624 - learning_rate: 1.0000e-03
Epoch 3/100
8857/8857 - 26s - 3ms/step - loss: 0.0705 - val_loss: 0.0653 - learning_rate: 1.0000e-03
Epoch 4/100
8857/8857 - 26s - 3ms/step - loss: 0.0661 - val_loss: 0.0787 - learning_rate: 1.0000e-03
Epoch 5/100
8857/8857 - 26s - 3ms/step - loss: 0.0631 - val_loss: 0.0862 - learning_rate: 1.0000e-03
Epoch 6/100

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
8857/8857 - 25s - 3ms/step - loss: 0.0610 - val_loss: 0.0845 - learning_rate: 1.0000e-03
Epoch 7/100
8857/8857 - 26s - 3ms/step - loss: 0.0557 - val_loss: 0.0873 - learning_rate: 5.0000e-04
Epoch 8/100
8857/8857 - 25s - 3ms/step - loss: 0.0551 - val_loss: 0.0943 - learning_rate: 5.0000e-04
Epoch 9/100
8857/8857 - 25s - 3ms/step - loss: 0.0544 - val_loss: 0.1000 - learn

In [ ]:

#@title 8) Precision@K curves & Top-K table for a chosen scoring
CHOSEN = "p99"  # "mean" or "p99"
y_true = A["label"].astype(int).values
y_score = -A[CHOSEN].values  # NEGATED
order = np.argsort(-y_score)
y_sorted = y_true[order]
ids_sorted = A.index.values[order]

Ks = np.arange(10, min(len(y_sorted), 3000)+1, 10)
prec_at_k = np.array([y_sorted[:k].mean() for k in Ks])
rec_at_k  = np.array([y_sorted[:k].sum()/y_true.sum() if y_true.sum()>0 else 0 for k in Ks])
f1_at_k   = 2*(prec_at_k*rec_at_k)/(prec_at_k+rec_at_k+1e-12)
bestK_idx = np.argmax(f1_at_k); best_K = int(Ks[bestK_idx])
print(f"Best K by F1@K: {best_K} | P={prec_at_k[bestK_idx]:.4f} | R={rec_at_k[bestK_idx]:.4f} | F1={f1_at_k[bestK_idx]:.4f}")

plt.figure(figsize=(6,4))
plt.plot(Ks, prec_at_k); plt.xlabel("K"); plt.ylabel("Precision@K"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/precision_at_k.png", dpi=150); plt.close()

plt.figure(figsize=(6,4))
plt.plot(Ks, rec_at_k); plt.xlabel("K"); plt.ylabel("Recall@K"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/recall_at_k.png", dpi=150); plt.close()

top_table = pd.DataFrame({
    "customer_id": ids_sorted[:best_K],
    "score": y_score[order][:best_K],
    "label": y_sorted[:best_K]
})
top_table.to_csv("/content/top_suspects_bestK.csv", index=False)
top_table.head(10)


Best K by F1@K: 3000 | P=0.0560 | R=0.0465 | F1=0.0508


,customer_id,score,label
0,46FE6127366C892AAD980DCE95C64841,-0.00078,0
1,0B0938D56D1A04D588184DBA32B22484,-0.00078,0
2,A3379B6D3C0AD9A843ABA0D3638D6732,-0.00078,0
3,5D1B426C6800B724E514FB3F7BB6CAD0,-0.00078,0
4,A2E1EF145EE766D9555846B9B33BF5FF,-0.00078,0
5,A2FE93FF694CE7470DDB574A28BF48B0,-0.00078,0
6,A303C8D7DF96C3F048E709C5A92C79F1,-0.00078,0
7,EF94D086D70932AA0FDA26A4B41112DA,-0.00078,0
8,9E18A6BC3D00DED499BB123F9D73D5D7,-0.00078,0
9,1F141F6A088B0C1F4C0C1DA1188311C7,-0.00078,0


In [ ]:
!pip -q install --no-cache-dir reportlab==3.6.12

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 61.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:

#@title 9) Build English PDF report (SSL)
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from datetime import datetime

pdf_path = "/content/SSL_SGCC_Report_EN.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4,
                        leftMargin=1.8*cm, rightMargin=1.8*cm,
                        topMargin=1.6*cm, bottomMargin=1.6*cm)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="H1", fontSize=16, leading=20, alignment=1, spaceAfter=10))
styles.add(ParagraphStyle(name="H2", fontSize=13, leading=17, spaceBefore=8, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontSize=10.5, leading=14))

flow=[]
flow.append(Paragraph("Self-Supervised Learning for Electricity-Theft Detection (SGCC)", styles["H1"]))
flow.append(Paragraph(datetime.now().strftime("%Y-%m-%d %H:%M"), styles["Body"]))
flow.append(Spacer(1,6))
flow.append(Paragraph("Denoising autoencoder trained on 60-day windows (stride 15) with per-customer z-score; client score is NEGATED aggregator of window MSE.", styles["Body"]))
flow.append(Spacer(1,8))

kpi = [
    ["Chosen client scoring", f"NEGATED {CHOSEN}"],
    ["NEGATED mean  (ROC/PR/F1)", f"{m_res['roc']:.4f} / {m_res['pr']:.4f} / {m_res['F1']:.4f}"],
    ["NEGATED p99   (ROC/PR/F1)", f"{p99_res['roc']:.4f} / {p99_res['pr']:.4f} / {p99_res['F1']:.4f}"],
    ["Best K by F1@K", f"{best_K} (P={prec_at_k[bestK_idx]:.4f}, R={rec_at_k[bestK_idx]:.4f})"],
]
tab = Table(kpi, colWidths=[7*cm, 7*cm])
tab.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
    ("BOX",(0,0),(-1,-1),1,colors.black),
    ("INNERGRID",(0,0),(-1,-1),0.4,colors.grey),
]))
flow.append(tab); flow.append(Spacer(1,8))

for title, fpath, wh in [
    ("Training Curve", "/content/figs/training_curve.png", (15*cm,8*cm)),
    ("Distribution of Window Errors", "/content/figs/window_err_hist.png", (15*cm,8*cm)),
    ("Distribution of Client Scores", "/content/figs/customer_score_hist.png", (15*cm,8*cm)),
    ("Precision@K", "/content/figs/precision_at_k.png", (15*cm,8*cm)),
    ("Recall@K", "/content/figs/recall_at_k.png", (15*cm,8*cm)),
]:
    if os.path.exists(fpath):
        flow.append(Paragraph(title, styles["H2"]))
        flow.append(Image(fpath, width=wh[0], height=wh[1]))
        flow.append(Spacer(1,6))

flow.append(PageBreak())
flow.append(Paragraph("Top suspects (best-K list)", styles["H2"]))
head = [["Rank","Customer ID","Score","Label"]]
rows = [[i+1, r["customer_id"], f"{r['score']:.6f}", int(r["label"])]
        for i, r in top_table.head(50).iterrows()]
tab2 = Table(head+rows, colWidths=[2*cm, 8*cm, 3*cm, 1.5*cm])
tab2.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
    ("BOX",(0,0),(-1,-1),1,colors.black),
    ("INNERGRID",(0,0),(-1,-1),0.4,colors.grey),
]))
flow.append(tab2)

doc.build(flow)
print("PDF saved:", pdf_path)


PDF saved: /content/SSL_SGCC_Report_EN.pdf


In [ ]:

#@title 10) Semi-supervised: train AE → embeddings → grid-search → test
from sklearn.preprocessing import StandardScaler

def build_client_embeddings(ae_model, panel_z, window=60, stride=15):
    X_all, idx_all = build_windows(panel_z, window=window, stride=stride)
    latent_model = keras.Model(ae_model.input, ae_model.get_layer('latent').output)
    Z_all = latent_model.predict(X_all, verbose=0)
    idx_df = pd.DataFrame(idx_all, columns=["customer_id","window_start"])
    Z_df = pd.DataFrame(Z_all)
    Z_df["customer_id"] = idx_df["customer_id"].values
    emb_mean = Z_df.groupby("customer_id").mean()
    emb_p95  = Z_df.groupby("customer_id").quantile(0.95)
    feats = emb_mean.add_suffix("_mean").join(emb_p95.add_suffix("_p95"), how="inner")
    feats.index = feats.index.astype(str)
    return feats

lab = cust_labels.copy()
lab["customer_id"] = lab["customer_id"].astype(str)
lab["label"] = lab["label"].astype(int)
lab_idx = lab.set_index("customer_id")["label"]

train_ids, test_ids = train_test_split(lab["customer_id"], test_size=0.2, stratify=lab["label"], random_state=SEED)
train_ids, val_ids  = train_test_split(train_ids, test_size=0.2, stratify=lab.set_index("customer_id").loc[train_ids, "label"], random_state=SEED)

ae_ss = autoenc  # reuse current AE
client_feats = build_client_embeddings(ae_ss, PANEL_Z, window=WINDOW, stride=STRIDE)

scaler = StandardScaler()
client_feats_std = pd.DataFrame(
    scaler.fit_transform(client_feats.values),
    index=client_feats.index, columns=client_feats.columns
)

def split_feats(feats, ids_tr, ids_va, ids_te):
    X_tr = feats.loc[feats.index.intersection(ids_tr)]
    X_va = feats.loc[feats.index.intersection(ids_va)]
    X_te = feats.loc[feats.index.intersection(ids_te)]
    y_tr = lab_idx.loc[X_tr.index].values
    y_va = lab_idx.loc[X_va.index].values
    y_te = lab_idx.loc[X_te.index].values
    return X_tr, y_tr, X_va, y_va, X_te, y_te

X_tr, y_tr, X_va, y_va, X_te, y_te = split_feats(client_feats_std, train_ids, val_ids, test_ids)

grid_results = []
best = None; best_va_pr = -1

# Logistic Regression
for C in [0.3, 1.0, 3.0]:
    logreg = LogisticRegression(class_weight="balanced", C=C, max_iter=500, random_state=SEED)
    logreg.fit(X_tr, y_tr)
    va_score = logreg.predict_proba(X_va)[:,1]
    pr = average_precision_score(y_va, va_score); roc = roc_auc_score(y_va, va_score)
    grid_results.append(("logreg", {"C":C}, pr, roc))
    if pr > best_va_pr:
        best_va_pr = pr; best = ("logreg", {"C":C})

# MLP
for hidden in [(64,), (128,)]:
    for alpha in [1e-4, 1e-3]:
        mlp = MLPClassifier(hidden_layer_sizes=hidden, alpha=alpha, random_state=SEED, max_iter=500)
        mlp.fit(X_tr, y_tr)
        va_score = mlp.predict_proba(X_va)[:,1]
        pr = average_precision_score(y_va, va_score); roc = roc_auc_score(y_va, va_score)
        grid_results.append(("mlp", {"hidden":hidden, "alpha":alpha}, pr, roc))
        if pr > best_va_pr:
            best_va_pr = pr; best = ("mlp", {"hidden":hidden, "alpha":alpha})

print("Best on VAL PR-AUC:", best, "PR=", best_va_pr)

# Refit on train+val, evaluate on test
X_trv = client_feats_std.loc[client_feats_std.index.intersection(set(train_ids).union(set(val_ids)))]
y_trv = lab_idx.loc[X_trv.index].values

if best[0] == "logreg":
    clf = LogisticRegression(class_weight="balanced", C=best[1]["C"], max_iter=500, random_state=SEED)
else:
    clf = MLPClassifier(hidden_layer_sizes=best[1]["hidden"], alpha=best[1]["alpha"], random_state=SEED, max_iter=500)

clf.fit(X_trv, y_trv)
test_proba = clf.predict_proba(X_te)[:,1]

def summarize_scores(y_true, y_score):
    roc = roc_auc_score(y_true, y_score)
    pr  = average_precision_score(y_true, y_score)
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    f1 = 2*(prec*rec)/(prec+rec+1e-12)
    best = np.nanargmax(f1)
    best_thr = thr[best-1] if 0<best<len(thr)+1 else np.quantile(y_score, 0.975)
    y_pred = (y_score >= best_thr).astype(int)
    P, R, F1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return dict(roc_auc=roc, pr_auc=pr, best_thr=float(best_thr), P=P, R=R, F1=F1)

semi_test = summarize_scores(y_te, test_proba)
print("Semi-supervised TEST metrics:", semi_test)


Best on VAL PR-AUC: ('mlp', {'hidden': (64,), 'alpha': 0.0001}) PR= 0.17135052907042325
Semi-supervised TEST metrics: {'roc_auc': np.float64(0.667520703615096), 'pr_auc': np.float64(0.17962202272406042), 'best_thr': 0.13878222927103637, 'P': 0.18694158075601375, 'R': 0.37621023513139695, 'F1': 0.2497704315886134}


In [ ]:

#@title 11) Precision@K (TEST) + Top-K CSV + EN PDF (semi-supervised)
order = np.argsort(-test_proba)
y_sorted = y_te[order]
ids_sorted = X_te.index.values[order]
Ks = np.arange(10, min(len(y_sorted), 3000)+1, 10)
prec_at_k = np.array([y_sorted[:k].mean() for k in Ks])
rec_at_k  = np.array([y_sorted[:k].sum()/y_te.sum() if y_te.sum()>0 else 0 for k in Ks])
f1_at_k   = 2*(prec_at_k*rec_at_k)/(prec_at_k+rec_at_k+1e-12)
bestK_idx = np.argmax(f1_at_k); best_K = int(Ks[bestK_idx])

plt.figure(figsize=(6,4))
plt.plot(Ks, prec_at_k); plt.xlabel("K"); plt.ylabel("Precision@K (TEST)"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/precision_at_k_test.png", dpi=150); plt.close()

plt.figure(figsize=(6,4))
plt.plot(Ks, rec_at_k); plt.xlabel("K"); plt.ylabel("Recall@K (TEST)"); plt.grid(True); plt.tight_layout()
plt.savefig("/content/figs/recall_at_k_test.png", dpi=150); plt.close()

topK = pd.DataFrame({"customer_id": ids_sorted[:best_K], "score": test_proba[order][:best_K], "label": y_sorted[:best_K]})
topK.to_csv("/content/top_suspects_test_bestK.csv", index=False)

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle, PageBreak
from datetime import datetime

pdf_path = "/content/SSL_SGCC_Semisup_EN.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4,
                        leftMargin=1.8*cm, rightMargin=1.8*cm,
                        topMargin=1.6*cm, bottomMargin=1.6*cm)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="H1", fontSize=16, leading=20, alignment=1, spaceAfter=10))
styles.add(ParagraphStyle(name="H2", fontSize=13, leading=17, spaceBefore=8, spaceAfter=6))
styles.add(ParagraphStyle(name="Body", fontSize=10.5, leading=14))

flow=[]
flow.append(Paragraph("Semi-Supervised SSL for Electricity-Theft Detection (SGCC)", styles["H1"]))
flow.append(Paragraph(datetime.now().strftime("%Y-%m-%d %H:%M"), styles["Body"]))
flow.append(Spacer(1,6))
flow.append(Paragraph("AE latent embeddings (mean+p95) → standardized features → classifier (grid-searched by validation PR-AUC).", styles["Body"]))
flow.append(Spacer(1,8))

kpi = [
    ["Best classifier (VAL)", str(best)],
    ["TEST ROC-AUC / PR-AUC", f"{semi_test['roc_auc']:.4f} / {semi_test['pr_auc']:.4f}"],
    ["Best-F1 threshold (TEST)", f"{semi_test['best_thr']:.6f} (P={semi_test['P']:.4f}, R={semi_test['R']:.4f}, F1={semi_test['F1']:.4f})"],
    ["Best K by F1@K (TEST)", f"{best_K} (P={prec_at_k[bestK_idx]:.4f}, R={rec_at_k[bestK_idx]:.4f})"],
]
tab = Table(kpi, colWidths=[7*cm, 7*cm])
tab.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
    ("BOX",(0,0),(-1,-1),1,colors.black),
    ("INNERGRID",(0,0),(-1,-1),0.4,colors.grey),
]))
flow.append(tab); flow.append(Spacer(1,8))

for title, fpath, wh in [
    ("Precision@K (TEST)", "/content/figs/precision_at_k_test.png", (15*cm,8*cm)),
    ("Recall@K (TEST)", "/content/figs/recall_at_k_test.png", (15*cm,8*cm)),
]:
    if os.path.exists(fpath):
        flow.append(Paragraph(title, styles["H2"]))
        flow.append(Image(fpath, width=wh[0], height=wh[1]))
        flow.append(Spacer(1,6))

flow.append(PageBreak())
flow.append(Paragraph("Top suspected clients (TEST, best-K)", styles["H2"]))
head = [["Rank","Customer ID","Score","Label"]]
rows = [[i+1, r["customer_id"], f"{r['score']:.6f}", int(r["label"])] for i, r in topK.iterrows()]
tab2 = Table(head+rows, colWidths=[2*cm, 8*cm, 3*cm, 1.5*cm])
tab2.setStyle(TableStyle([
    ("BACKGROUND",(0,0),(-1,0),colors.whitesmoke),
    ("BOX",(0,0),(-1,-1),1,colors.black),
    ("INNERGRID",(0,0),(-1,-1),0.4,colors.grey),
]))
flow.append(tab2)

doc.build(flow)
print("PDF saved:", pdf_path)


PDF saved: /content/SSL_SGCC_Semisup_EN.pdf


In [ ]:

#@title 12) Snapshot /content to Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')

TS = time.strftime("%Y-%m-%d_%H-%M-%S")
TARGET_DIR = f"/content/drive/MyDrive/sgcc_project_snapshots/snapshot_{TS}"
os.makedirs(TARGET_DIR, exist_ok=True)
!rsync -av --exclude='drive' --exclude='sample_data' /content/ "{TARGET_DIR}/"
print("Snapshot saved to:", TARGET_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
sending incremental file list
./
SSL_SGCC_Report_EN.pdf
SSL_SGCC_Semisup_EN.pdf
ae_train_history.csv
client_aggregates.csv
data.csv
ssl_eval.json
top_suspects_bestK.csv
top_suspects_test_bestK.csv
window_errors.csv
.config/
.config/.last_opt_in_prompt.yaml
.config/.last_survey_prompt.yaml
.config/.last_update_check.json
.config/active_config
.config/config_sentinel
.config/default_configs.db
.config/gce
.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
.config/configurations/
.config/configurations/config_default
.config/logs/
.config/logs/2025.12.09/
.config/logs/2025.12.09/14.40.47.605300.log
.config/logs/2025.12.09/14.41.18.717681.log
.config/logs/2025.12.09/14.41.27.893750.log
.config/logs/2025.12.09/14.41.33.792924.log
.config/logs/2025.12.09/14.41.42.675750.log
.config/logs/2025.12.09/14.41.43.412452.log
ElectricityTheftDetection/
E